# Fase 1: Adquisición de Datos (Estrategia Mixta)
Este Notebook documenta el paso inicial del proyecto modular de Cashback. El objetivo es extraer de forma verídica entre 30 y 40 marcas con sus ofertas actuales mediante **Web Scraping**.

### Conceptos Clave para Repasar:
* **`requests`**: Librería para realizar peticiones HTTP (descargar el código HTML de una web).
* **`BeautifulSoup`**: Librería para parsear(analizar y desglosar una cadena de datos o texto para extraer info útil, convertirla a formato estructurado o traducir a instrucciones que un sistema puede entender y procesar) y navegar por el árbol HTML de una página web mediante selectores.
* **`User-Agent`**: Cabecera HTTP que identifica al cliente (nuestro script) ante el servidor. Lo usamos para simular un navegador real y evitar bloqueos automáticos.


In [10]:
# Importamos las librerías necesarias para esta fase
import os
import csv
import requests
from bs4 import BeautifulSoup

print("Librerías cargadas correctamente en el entorno virtual.")


Librerías cargadas correctamente en el entorno virtual.


## Paso 1.1: Descarga del HTML
Para comunicarnos con la web objetivo, enviamos un encabezado (`headers`) que simula una visita desde un Mac con Google Chrome. Esto previene que el servidor nos rechace por parecer un bot genérico.


In [11]:
def descargar_html(url):
    # Simulamos un navegador real en Mac OS
    headers = {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }
    
    try:
        respuesta = requests.get(url, headers=headers, timeout=10)
        if respuesta.status_code == 200:
            print("¡Conexión exitosa (Código 200)!")
            return respuesta.text
        else:
            print(f"Error en la descarga. Código de estado: {respuesta.status_code}")
            return None
    except Exception as e:
        print(f"Error de conexión fatal: {e}")
        return None

# Prueba conceptual con una URL ficticia o de pruebas (reemplazar por la real)
url_prueba = "https://httpbin.org" 
html_puro = descargar_html(url_prueba)


¡Conexión exitosa (Código 200)!


## Paso 1.2: Parseo y Extracción (Scraping)
Una vez que tenemos el HTML como texto plano, `BeautifulSoup` lo transforma en un objeto estructurado. 

* Usamos `.find_all()` para buscar los contenedores de las marcas.
* Usamos `.find()` para extraer el texto de las etiquetas específicas (nombres, porcentajes).
* Implementamos un bloque `try-except` interno para que, si una marca tiene datos incompletos, el script no rompa y continúe con la siguiente.

In [12]:
def extraer_ofertas(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    lista_marcas = []
    
    # IMPORTANTE: Modificar estos selectores cuando tengamos la estructura CSS real
    bloques_marcas = soup.find_all('div', class_='merchant-card') 
    
    # Limitamos a 40 para cumplir con el alcance del Paso A
    for bloque in bloques_marcas[:40]: 
        try:
            nombre = bloque.find('h3', class_='merchant-name').text.strip()
            cashback = bloque.find('span', class_='cashback-rate').text.strip()
            
            lista_marcas.append({
                "marca": nombre,
                "cashback_ofrecido": cashback,
                "pais": "ES"
            })
        except AttributeError:
            # Si un elemento no existe (ej. una marca sin cashback configurado), salta aquí y continúa
            continue 
            
    return lista_marcas

print("Función 'extraer_ofertas' definida y lista para su uso.")


Función 'extraer_ofertas' definida y lista para su uso.


## Paso 1.3: Almacenamiento en data/raw/
Cumpliendo con la arquitectura profesional impuesta, los datos extraídos se guardarán en su formato nativo "sucio" dentro de `data/raw/`. 

Usamos `os.path.join` para que las rutas funcionen de forma nativa tanto en sistemas Unix (tu Mac) como en Windows, garantizando la portabilidad del código.


In [13]:
def guardar_datos_raw(datos, nombre_archivo="marcas_raw.csv"):
    # Construimos la ruta relativa: data/raw/marcas_raw.csv
    ruta_destino = os.path.join("..", "data", "raw", nombre_archivo) # El ".." sube un nivel desde la carpeta notebooks/
    
    # Creamos el directorio de forma segura si no existiera
    os.makedirs(os.path.dirname(ruta_destino), exist_ok=True)
    
    if not datos:
        print("Atención: No hay datos en la lista para exportar.")
        return
        
    columnas = datos[0].keys() # Extrae las claves del diccionario como cabeceras del CSV
    
    with open(ruta_destino, mode="w", encoding="utf-8", newline="") as archivo:
        escritor = csv.DictWriter(archivo, fieldnames=columnas)
        escritor.writeheader()
        escritor.writerows(datos)
        
    print(f"📁 Archivo guardado con éxito en: {ruta_destino}")

print("Función 'guardar_datos_raw' integrada.")


Función 'guardar_datos_raw' integrada.


## Paso 1.4: Orquestación y Prueba General
Unimos todas las funciones anteriores para ejecutar el flujo completo de adquisición de datos crudos.


In [14]:
# URL final de la plataforma o red de afiliación que vas a mapear
url_definitiva = "https://ejemplo-plataforma.com" 

html_final = descargar_html(url_definitiva)

if html_final:
    resultados = extraer_ofertas(html_final)
    print(f"Total de marcas capturadas: {len(resultados)}")
    guardar_datos_raw(resultados)
else:
    print("No se pudo iniciar el proceso de scraping.")


Error de conexión fatal: HTTPSConnectionPool(host='ejemplo-plataforma.com', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPSConnection(host='ejemplo-plataforma.com', port=443): Failed to resolve 'ejemplo-plataforma.com' ([Errno 8] nodename nor servname provided, or not known)"))
No se pudo iniciar el proceso de scraping.


## Paso 1.5: Inspección de Selectores Reales (Guía de Estudio)
Para que el método `BeautifulSoup.find_all()` funcione, debemos decirle exactamente en qué etiquetas de la página web están los datos.

### Pasos para hacerlo en el navegador (Chrome/Safari en Mac):
1. Entra en la página web de ofertas o cashback que vas a raspar.
2. Haz clic derecho sobre el **nombre de una marca** y selecciona **Inspeccionar**.
3. Busca la etiqueta HTML (suele ser un `<div>`, `<li>` o `<a>`) que envuelve a **toda la tarjeta** de la tienda. Copia su clase (`class="..."`).
4. Haz lo mismo para el **nombre de la tienda** (ej. `<h3>` con su clase) y el **porcentaje de cashback** (ej. `<span>` o `<div>` con su clase).


In [15]:
def extraer_ofertas_reales(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    lista_marcas = []
    
    # =========================================================================
    # CONFIGURACIÓN DE SELECTORES (Modifica esto tras inspeccionar la web)
    # =========================================================================
    CLASE_CONTENEDOR_PADRE = "merchant-card"  # El bloque entero de la tienda
    CLASE_NOMBRE_TIENDA     = "merchant-name"  # La etiqueta con el nombre
    CLASE_VALOR_CASHBACK    = "cashback-rate"  # La etiqueta con el % o dinero
    # =========================================================================
    
    # Buscamos todos los bloques de tiendas en la página
    bloques_marcas = soup.find_all(class_=CLASE_CONTENEDOR_PADRE) 
    
    if not bloques_marcas:
        print("⚠️ Alerta: No se encontraron elementos con la clase contenedora configurada.")
        return []
    
    # Extraemos el texto limpiando espacios en blanco innecesarios
    for bloque in bloques_marcas[:40]: # Límite estricto de 30-40 marcas
        try:
            # Buscamos las etiquetas hijas dentro del bloque de la tienda
            nombre_elemento = bloque.find(class_=CLASE_NOMBRE_TIENDA)
            cashback_elemento = bloque.find(class_=CLASE_VALOR_CASHBACK)
            
            # Verificamos que existan antes de extraer el texto (.text)
            if nombre_elemento and cashback_elemento:
                nombre = nombre_elemento.text.strip()
                cashback = cashback_elemento.text.strip()
                
                lista_marcas.append({
                    "marca": nombre,
                    "cashback_ofrecido": cashback,
                    "pais": "ES" # Forzamos el país de origen para el análisis
                })
        except Exception as e:
            print(f"Error al procesar una tarjeta individual: {e}")
            continue 
            
    print(f"✔ Extracción finalizada. Se han capturado {len(lista_marcas)} marcas reales.")
    return lista_marcas


In [16]:
# Introduce aquí la URL real de la lista de tiendas/marcas que vas a usar
URL_OBJETIVO_REAL = "https://ejemplo.com" 

html_puro = descargar_html(URL_OBJETIVO_REAL)

if html_puro:
    # Ejecutamos la extracción con los selectores configurados
    datos_finales = extraer_ofertas_reales(html_puro)
    
    # Si capturamos datos, los guardamos directamente en la carpeta raw
    if datos_finales:
        guardar_datos_raw(datos_finales, nombre_archivo="marcas_raw.csv")
else:
    print("❌ No se pudo descargar el HTML. Revisa la URL o tu conexión a internet.")


¡Conexión exitosa (Código 200)!
⚠️ Alerta: No se encontraron elementos con la clase contenedora configurada.


## Paso 1.6: Extracción Multi-Plataforma
En un entorno de producción real, una plataforma de cashback consulta múltiples fuentes (APIs/Scraping) simultáneamente. 
Para simular este comportamiento con datos verídicos y robustos, mapearemos un diccionario con las URLs de origen de cada red de afiliación y procesaremos sus ofertas mediante un bucle unificado.


In [17]:
def extraer_datos_de_red(html_content, nombre_red):
    """Parsea el HTML adaptando los selectores según la red de afiliación origen."""
    soup = BeautifulSoup(html_content, 'html.parser')
    lista_ofertas = []
    
    # Diccionario de selectores simulados basados en las estructuras comunes de cada red
    # Al investigar las webs reales, solo debes actualizar estos valores en el diccionario
    config_redes = {
        "Awin": {"padre": "awin-merchant", "nombre": "title", "tasa": "commission-rate"},
        "CJ_Affiliate": {"padre": "cj-advertiser-card", "nombre": "name", "tasa": "payout"},
        "Rakuten": {"padre": "rakuten-brand", "nombre": "brand-name", "tasa": "cashback-value"},
        "Impact": {"padre": "impact-brand-card", "nombre": "brand-title", "tasa": "reward"}
    }
    
    cfg = config_redes.get(nombre_red)
    if not cfg:
        return []
        
     bloques = soup.find_all(class_=cfg["padre"])
    
    for bloque in bloques[:10]:  # Tomamos 10 marcas de cada red para sumar las 40 finales
        try:
            nombre = bloque.find(class_=cfg["nombre"]).text.strip()
            cashback = bloque.find(class_=cfg["tasa"]).text.strip()
            
            lista_ofertas.append({
                "marca": nombre,
                "cashback_ofrecido": cashback,
                "red_origen": nombre_red,
                "pais": "ES"
            })
        except AttributeError:
            continue
            
    return lista_ofertas


IndentationError: unindent does not match any outer indentation level (<string>, line 19)

In [ ]:
# Mapeo de URLs reales/objetivo para cada red de afiliados
urls_fuentes = {
    "Awin": "https://ejemplo.com",
    "CJ_Affiliate": "https://ejemplo.com",
    "Rakuten": "https://ejemplo.com",
    "Impact": "https://ejemplo.com"
}

dataset_consolidado = []

for red, url in urls_fuentes.items():
    print(f"Iniciando raspado en la red: {red}...")
    html = descargar_html(url) # Reutiliza la función de la Celda 4
    
    if html:
        ofertas_red = extraer_datos_de_red(html, red)
        dataset_consolidado.extend(ofertas_red)
        print(f"✔ Capturadas {len(ofertas_red)} marcas desde {red}.")
    else:
        print(f"⚠️ Saltando {red} por fallo en la descarga.")

print(f"\n🚀 Proceso completado. Total de marcas consolidadas: {len(dataset_consolidado)}")

# Guardamos los resultados finales en la carpeta de datos crudos
if dataset_consolidado:
    guardar_datos_raw(dataset_consolidado, nombre_archivo="marcas_multi_red_raw.csv")


## Práctica Guiada: Mi Primer Scraping Real

### Paso 1: Inspección Visual y Teórica
Vamos a raspar la web `https://scrapethissit.com`. 
Al inspeccionar esta web con el clic derecho del navegador, descubriremos la estructura que se repite para cada elemento:
1. El **Contenedor Padre** (la fila o tarjeta de cada elemento) usa la etiqueta `<tr class="team">`.
2. El **Nombre** del elemento está dentro de un `<td class="name">`.
3. El **Valor o Tasa** (que para nuestro proyecto simulará ser el cashback) está dentro de un `<td class="wins">`.


In [ ]:
def extraer_ofertas_igraal_real(html_content):
    soup = BeautifulSoup(html_content, 'html.parser')
    lista_marcas = []
    
    # 1. Definimos la clase del CONTENEDOR PADRE que hemos visto en tu captura.
    # En webs modernas con Next.js (como iGraal), basta con usar las primeras clases estables
    CLASE_PADRE = "nalu verticalbasecardlite"
    
    # 2. Buscamos todas las tarjetas de tiendas que hay en la página principal
    # Usamos una función lambda para buscar elementos que CONTENGAN esa clase
    tarjetas = soup.find_all('div', class_=lambda c: c and CLASE_PADRE in c)
    
    print(f"Buscando tarjetas... ¡Se han detectado {len(tarjetas)} tiendas en el HTML!")
    
    for tarjeta in tarjetas[:40]: # Límite para cumplir los requisitos de la escuela
        try:
            # Buscamos los textos dentro de la tarjeta. 
            # iGraal suele estructurar el nombre y el % en etiquetas de texto plano internas
            textos = [t.strip() for t in tarjeta.stripped_strings]
            
            # Si la tarjeta tiene la estructura esperada de textos (ej: ['En línea', 'Uber', '3% de reembolso'])
            if len(textos) >= 3:
                # Buscamos cuál de los textos contiene el símbolo '%' o la palabra 'reembolso'
                cashback = "No disponible"
                nombre = textos[1] # Por estructura, el segundo texto suele ser el nombre de la marca
                
                for texto in textos:
                    if "%" in texto or "reembolso" in texto.lower():
                        cashback = texto
                        break
                
                lista_marcas.append({
                    "marca": nombre,
                    "cashback_ofrecido": cashback,
                    "pais": "ES"
                })
        except Exception as e:
            continue
            
    return lista_marcas


In [ ]:
import requests
import pandas as pd
import os

url_igraal = "https://igraal.com"
headers = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

print("Descargando la página de inicio de iGraal...")
respuesta = requests.get(url_igraal, headers=headers, timeout=15)

if respuesta.status_code == 200:
    # Extraemos los datos usando la estructura que descubriste en el inspector
    datos_igraal = extraer_ofertas_igraal_real(respuesta.text)
    
    # Convertimos a DataFrame para visualizarlo bonito en tu pantalla de estudio
    df_marcas = pd.DataFrame(datos_igraal)
    
    if not df_marcas.empty:
        print("\n✔ ¡Scraping completado con éxito! Aquí tienes una muestra de tus datos reales:")
        display(df_marcas.head(10)) # Muestra las primeras 10 filas en el Notebook
        
        # Guardamos el archivo final en data/raw/
        ruta_guardado = os.path.join("..", "data", "raw", "marcas_igraal_raw.csv")
        df_marcas.to_csv(ruta_guardado, index=False, encoding="utf-8")
        print(f"\n📁 Archivo guardado con éxito en: {ruta_guardado}")
    else:
        print("⚠️ El HTML descargado no contenía los elementos. iGraal podría estar usando renderizado por JavaScript.")
else:
    print(f"Error de conexión: {respuesta.status_code}")
